In [3]:
import json
import statistics
from pathlib import Path

from tokenizers import Tokenizer

# ---- parameters ----
MIN_TOKENS = 512       # lower bound of a chunk (bge-m3 tokens)
TARGET_TOKENS = 768    # a chunk is closed as soon as it reaches this size
MAX_TOKENS = 1024      # upper bound
OVERLAP = 0.15         # share of the previous chunk repeated at the start of the next one

# Notebook lives in rag/build-samples/, corpus in <root>/corpus/
ROOT = Path.cwd().resolve()
while not (ROOT / "corpus" / "clean").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN_DIR = ROOT / "corpus" / "clean"
OUT_PATH = ROOT / "corpus" / "chunks" / "time_chunks.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

tokenizer = Tokenizer.from_pretrained("BAAI/bge-m3")   # same tokenizer as the embedder
count_tokens = lambda s: len(tokenizer.encode(s, add_special_tokens=False).ids)

docs = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(CLEAN_DIR.glob("*.json"))]
print(f"{len(docs)} streams from {CLEAN_DIR}")

25 streams from S:\ai_projects\demo_26_RAG-vs-agent\corpus\clean


In [4]:
def to_sec(tc: str) -> int:
    h, m, s = map(int, tc.split(":"))
    return h * 3600 + m * 60 + s


def split_entries(entries):
    """[{"00:00:28": "text"}, ...] -> (timecodes, texts)"""
    tcs, texts = [], []
    for e in entries:
        (tc, txt), = e.items()
        tcs.append(tc)
        texts.append(txt)
    return tcs, texts


def chunk_spans(toks: list[int]) -> list[tuple[int, int]]:
    """Greedy time-ordered chunking over entry token counts.

    Returns [(i, j), ...]: chunk = entries i..j-1. A chunk is closed once it reaches
    TARGET_TOKENS (never exceeding MAX_TOKENS), the next chunk starts OVERLAP * size
    tokens before the end of the previous one (entry granularity).
    """
    n = len(toks)
    spans: list[tuple[int, int]] = []
    i = 0
    while i < n:
        j, total = i, 0
        while j < n and total < TARGET_TOKENS:
            if total and total + toks[j] > MAX_TOKENS:
                break
            total += toks[j]
            j += 1
        spans.append((i, j))
        if j >= n:
            break
        # overlap = trailing entries whose token sum is closest to OVERLAP * size
        need, k, acc = OVERLAP * total, j, 0
        best_k, best_diff = j, None        # j = no overlap (only for single-entry chunks)
        while k > i + 1:                   # k > i + 1 guarantees progress
            k -= 1
            acc += toks[k]
            if best_diff is None or abs(acc - need) < best_diff:
                best_k, best_diff = k, abs(acc - need)
            if acc >= need:
                break
        i = best_k

    # the tail chunk can be too short: merge it into the previous one if it fits,
    # otherwise grow it backwards (more overlap) until it reaches MIN_TOKENS
    if len(spans) > 1 and sum(toks[spans[-1][0]:spans[-1][1]]) < MIN_TOKENS:
        (pi, _), (li, lj) = spans[-2], spans[-1]
        if sum(toks[pi:lj]) <= MAX_TOKENS:
            spans[-2:] = [(pi, lj)]
        else:
            while li > pi + 1 and sum(toks[li:lj]) < MIN_TOKENS:
                li -= 1
            spans[-1] = (li, lj)
    return spans


def chunk_doc(doc: dict) -> list[dict]:
    tcs, texts = split_entries(doc["text_clean"])
    toks = [count_tokens(t) for t in texts]
    duration = to_sec(doc["duration"])
    chunks = []
    for i, j in chunk_spans(toks):
        text = " ".join(texts[i:j])
        end = to_sec(tcs[j]) if j < len(tcs) else duration   # next entry start / end of stream
        chunks.append({
            "id": f"{doc['youtubeId']}#{tcs[i]}",   # stable: survives re-chunking with other params
            "youtubeId": doc["youtubeId"],
            "title": doc["title"],
            "date": doc["date"],
            "url": doc["url"],
            "start": to_sec(tcs[i]),
            "end": end,
            "start_tc": tcs[i],
            "n_entries": j - i,
            "tokens": count_tokens(text),            # real count of the joined text
            "text": text,
        })
    return chunks

In [5]:
all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_doc(doc))

OUT_PATH.write_text(json.dumps(all_chunks, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(all_chunks)} chunks -> {OUT_PATH}")

917 chunks -> S:\ai_projects\demo_26_RAG-vs-agent\corpus\chunks\time_chunks.json


In [6]:
# ---- Sanity checks ----
by_stream = {}
for c in all_chunks:
    by_stream.setdefault(c["youtubeId"], []).append(c)

assert len({c["id"] for c in all_chunks}) == len(all_chunks), "chunk ids are not unique"

overlaps = []
for d in docs:
    cs = by_stream[d["youtubeId"]]
    starts = [c["start"] for c in cs]
    assert starts == sorted(starts) and len(set(starts)) == len(starts), "starts must increase"
    assert cs[0]["start"] == to_sec(next(iter(d["text_clean"][0]))), "first entry not covered"
    assert cs[-1]["end"] == to_sec(d["duration"]), "stream tail not covered"
    for a, b in zip(cs, cs[1:]):
        assert b["start"] < a["end"], "gap between chunks"   # overlap, never a hole
        overlaps.append((a["end"] - b["start"]) / (a["end"] - a["start"]))

tok = [c["tokens"] for c in all_chunks]
dur = [c["end"] - c["start"] for c in all_chunks]
out = [c for c in all_chunks if not MIN_TOKENS <= c["tokens"] <= MAX_TOKENS]
print(f"chunks per stream: {min(map(len, by_stream.values()))}..{max(map(len, by_stream.values()))}")
print(f"tokens  min/median/max: {min(tok)} / {statistics.median(tok):.0f} / {max(tok)}")
print(f"seconds min/median/max: {min(dur)} / {statistics.median(dur):.0f} / {max(dur)}")
print(f"time overlap between neighbours (median): {statistics.median(overlaps):.1%}")
print(f"outside [{MIN_TOKENS}, {MAX_TOKENS}] tokens: {len(out)}")
for c in out[:5]:
    print("  ", c["id"], c["tokens"], "tokens,", c["n_entries"], "entries")

chunks per stream: 24..58
tokens  min/median/max: 534 / 807 / 1023
seconds min/median/max: 140 / 260 / 489
time overlap between neighbours (median): 12.4%
outside [512, 1024] tokens: 0


In [7]:
# ---- Example: two neighbouring chunks, to eyeball the overlap ----
a, b = all_chunks[3], all_chunks[4]
for c in (a, b):
    print(f"[{c['id']}] {c['start']}s-{c['end']}s, {c['tokens']} tokens")
print("\nEND of chunk 1  :", a["text"][-250:])
print("START of chunk 2:", b["text"][:250])

[wht8lf-h9N8#00:10:19] 619s-854s, 796 tokens
[wht8lf-h9N8#00:13:46] 826s-1065s, 866 tokens

END of chunk 1  : і, здається, так ж ця штука називалася, і я навіть зараз спробую на карті його знайти. Він був дуже поруч, від того місця, де я на той момент жив, тому було неважко туди попадать мені. так, якщо хтось Запоріжжя, то знає, що в Запоріжжі є Фестивальна.
START of chunk 2: трошки в іншу сторону, тобто тут це був центр молоді, здається, так ж ця штука називалася, і я навіть зараз спробую на карті його знайти. Він був дуже поруч, від того місця, де я на той момент жив, тому було неважко туди попадать мені. так, якщо хтос
